In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.eos_token_id

dataset = [
    {
        "text": "patient diagnosed diabetes received medicine hospital",
        "topic_words": ["patient", "doctor", "medicine", "hospital", "diabetes"],
        "summary": "patient received diabetes treatment"
    },
    {
        "text": "bank approved loan for customer",
        "topic_words": ["bank", "money", "loan", "finance", "customer"],
        "summary": "bank approved loan"
    }
]

def make_prompt(text, topic_words):
    topic_text = " ".join(topic_words)
    return f"Topic: {topic_text}\nText: {text}\nSummary:"

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

for epoch in range(20):
    total_loss = 0

    for item in dataset:
        prompt = make_prompt(item["text"], item["topic_words"])
        full_text = prompt + " " + item["summary"] + tokenizer.eos_token

        batch = tokenizer(
            full_text,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["input_ids"]
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss={total_loss:.4f}")

D:\anaconda3\envs\web_lumplt\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\anaconda3\envs\web_lumplt\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shiri\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: ht

Epoch 1, Loss=10.1948
Epoch 2, Loss=6.6757
Epoch 3, Loss=4.4005
Epoch 4, Loss=2.7884
Epoch 5, Loss=1.7219
Epoch 6, Loss=0.9875
Epoch 7, Loss=0.5014
Epoch 8, Loss=0.2303
Epoch 9, Loss=0.1527
Epoch 10, Loss=0.1243
Epoch 11, Loss=0.0816
Epoch 12, Loss=0.0918
Epoch 13, Loss=0.0973
Epoch 14, Loss=0.0828
Epoch 15, Loss=0.0732
Epoch 16, Loss=0.0780
Epoch 17, Loss=0.0802
Epoch 18, Loss=0.0773
Epoch 19, Loss=0.0725
Epoch 20, Loss=0.0704


In [2]:
model.eval()

text = "sick patient received medicine doctor"
topic_words = ["patient", "doctor", "medicine", "health", "treatment"]

prompt = make_prompt(text, topic_words)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

with torch.no_grad():
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=10,
        no_repeat_ngram_size=2,
        repetition_penalty=2.0,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

summary = output.split("Summary:")[-1].strip()

print(summary)

healthy user approved diabetes medication


GPT is generally better than BERT for summarization because GPT is naturally designed for text generation using autoregressive decoding.

However, encoder-decoder models, such as the Transformer architecture from *Attention Is All You Need*, are often more suitable for summarization because summarization requires BOTH:

- deep understanding of the input document
- fluent generation of the output summary

In encoder-decoder architectures:

```text
Encoder → understands the document
Decoder → generates the summary